In [ ]:
from pathlib import Path
import sys

# Make this package's scripts/ importable (exp_info, regions, data_utils, ...).
PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts" / "exp_info.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook from inside the polar_analysis package.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

In [ ]:
# Parameters: directories and settings for this run. Edit them for another
# machine or experiment, or override with papermill (-p NAME VALUE).
GLOBAL_DIR           = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper/data/global"  # data/global
GLOBAL_DRIFTCORR_DIR = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper/data/global_driftcorr"  # data/global_driftcorr
REGMN_DRIFTCORR_DIR  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper/data/regmn_driftcorr"  # data/regmn_driftcorr

In [16]:
import os
import glob
import re
import json
import time
import datetime
import xarray as xr
import xcdat as xc
import numpy as np
import pandas as pd
import shutil
import psutil
import subprocess
from pathlib import Path
from typing import Union
# --- safe imports ---
import warnings


In [17]:
def read_var_dict():
    var_dict = {
            'SST'         : {'unit': '$\circ$C',     'min': -6,  'max': 6,  'nlev': 11},
            'SICONC'      : {'unit': '%',            'min': -6,  'max': 6,  'nlev': 11},
            #'SIMASS'      : {'unit': 'kg m$^{-2}$',  'min': -6,  'max': 6,  'nlev': 11},
            #'SITHICK'  : {'unit': 'm',            'min': -6,  'max': 6,  'nlev': 11},
            'TS'      : {'unit': 'K',            'min': -6,  'max': 6,  'nlev': 11},
            'TURFLX'  : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11}, 
            'LHFLX'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'SHFLX'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'RADNET'  : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FSNS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FSDS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FSUS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FSNSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FSDSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FSUSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FLDS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FLNS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FLUS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FLUSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FLDSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'FLNSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'SWCF_SRF': {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'LWCF_SRF': {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'CRE_SRF' : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'SWCF'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'LWCF'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'CRE'     : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11},
            'ALBEDOC_SRF' : {'unit': '%',        'min': -6,  'max': 6,  'nlev': 13},
            'ALBEDO_SRF'  : {'unit': '%',        'min': -6,  'max': 6,  'nlev': 13},
        }
    return var_dict 

In [18]:
class EnsembleDataProcs:
    def __init__(
        self,
        data_dir,
        file_template,          # e.g., "exp.en%(ENS).monthly_base.YYYYMM-YYYYMM.nc"
        num_ens,
        var_dict,
        out_dir=None,
        chunks=None,
        fontz=12,
        keep_vars=None,
        lat_name="lat",
        lon_name="lon",
        region_defs=None,       # optional custom regions; else uses get_default_region()
    ):
        self.data_dir    = data_dir
        self.out_dir     = out_dir if out_dir else os.path.join(data_dir, "out")
        self.file_tmpl   = file_template
        self.num_ens     = int(num_ens)
        self.var_dict    = var_dict or {}
        self.fontz       = fontz
        self.chunks      = chunks
        self.ds_all      = None
        self.time_coord  = None
        self.keep_vars   = set(keep_vars or ["area"])
        self.lat_name    = lat_name
        self.lon_name    = lon_name
        self.region_defs = region_defs or self.get_default_region()

        # Normalize region longitudes once & keep raw endpoints for full-ring detection
        self._normalize_region_defs_inplace()

        Path(self.out_dir).mkdir(parents=True, exist_ok=True)

    # ---------- Regions ----------
    def get_default_region(self):
        # You can define in either 0..360 or -180..180; we normalize internally
        return {
            "global":   [(-90,  90), (-180, 180)],
            "Atlantic": [(  0,  60), ( 265, 345)],  # will normalize to [-95, -15]
            "Arctic":   [( 50,  90), (-180, 180)],
            "Arctic2":  [( 66.5,90), (-180, 180)],
            "CONUS":    [( 25,  50), ( 235, 295)],
            "Antarctic":[(-90, -50), (-180, 180)],
            "PolarN":   [( 50,  90), (-180, 180)],
            "Greenland":[( 55,  90), ( -75, -10)],
        }

    def _normalize_lon_val(self, x: float) -> float:
        """Normalize a single longitude to [-180, 180)."""
        return ((x + 180.0) % 360.0) - 180.0

    def _normalize_lon_pair(self, lon0: float, lon1: float):
        """
        Normalize a (lon0, lon1) pair conditionally:
          - If both endpoints are in [-180, 180], leave them as-is.
          - If both endpoints are in [0, 360], convert both to [-180, 180).
          - Otherwise (mixed/out-of-range), normalize both (preserves intent for wrap-around).
        """
        in_0_360    = lambda v: 0.0    <= v <= 360.0
        in_m180_180 = lambda v: -180.0 <= v <= 180.0

        both_180 = in_m180_180(lon0) and in_m180_180(lon1)
        both_360 = in_0_360(lon0)    and in_0_360(lon1)

        if both_180:
            return float(lon0), float(lon1)
        if both_360:
            return self._normalize_lon_val(lon0), self._normalize_lon_val(lon1)

        # Mixed/out-of-range: normalize both for consistent semantics
        warnings.warn(f"Mixed/out-of-range longitude pair ({lon0}, {lon1}) "
                      f"— normalizing both to [-180, 180).")
        return self._normalize_lon_val(lon0), self._normalize_lon_val(lon1)

    def _normalize_region_defs_inplace(self):
        """
        Normalize region longitudes only when needed, and stash raw endpoints
        to detect full-ring boxes (e.g., 0..360) later.
        """
        normed = {}
        meta = {}
        for name, ((lat0, lat1), (lon0, lon1)) in self.region_defs.items():
            lon0n, lon1n = self._normalize_lon_pair(float(lon0), float(lon1))
            normed[name] = [(float(lat0), float(lat1)), (lon0n, lon1n)]
            meta[name]   = {"lon_raw": (float(lon0), float(lon1))}
        self.region_defs = normed
        self._region_meta = meta

    def _region_segments_lon(self, lon0n: float, lon1n: float, lon0_raw: float, lon1_raw: float):
        """
        Return lon segments in [-180, 180) covering the requested box.
        Detect full-ring using raw endpoints (difference multiple of 360 and not equal).
        """
        full_ring = (np.isclose(((lon1_raw - lon0_raw) % 360.0), 0.0) and
                     not np.isclose(lon1_raw, lon0_raw))
        if full_ring:
            return [(-180.0, 180.0)]
        if lon0n <= lon1n:
            return [(lon0n, lon1n)]
        else:
            return [(lon0n, 180.0), (-180.0, lon1n)]

    @staticmethod
    def _normalize_dataset_longitude(ds, lon_name="lon"):
        lons = np.asarray(ds[lon_name].values)
        if np.nanmin(lons) >= 0 and np.nanmax(lons) > 180:  # clearly 0..360
            ds = ds.assign_coords({lon_name: ((lons + 180) % 360 - 180)})
            ds = ds.sortby(lon_name)
        return ds

    # ---------- File discovery ----------
    def _ensemble_paths(self):
        """
        Supports templates with '%(ENS)' (e.g., '.en%(ENS).') or without (single file).
        Auto-detects whether ensembles are indexed en00.. or en01..
        """
        paths = []
        has_placeholder = "%(ENS)" in self.file_tmpl
        if has_placeholder:
            # try 1..N first (en01..enNN), else fall back to 0..N-1 (en00..)
            for start, stop in ((0, self.num_ens), (0, self.num_ens)):
                trial = []
                for i in range(start, stop):
                    ens_str = f"{i:02d}"
                    fname = self.file_tmpl.replace("%(ENS)", ens_str)
                    fpath = os.path.join(self.data_dir, fname)
                    if os.path.isfile(fpath):
                        trial.append(fpath)
                    print(self.data_dir, fname)
                if trial:
                    paths = trial
                    break
        else:
            fpath = os.path.join(self.data_dir, self.file_tmpl)
            if os.path.isfile(fpath):
                paths.append(fpath)
            print(self.data_dir, os.path.basename(fpath))

        if not paths:
            raise FileNotFoundError("No ensemble files found after filtering missing files.")
        return sorted(paths)

    # ---------- Basic preprocessing ----------
    def _preprocess(self, ds: xr.Dataset) -> xr.Dataset:
        keep_data = list(ds.data_vars)
        keep_aux  = [v for v in self.keep_vars if v in ds]  # e.g., 'area' or 'areacella'
        out = ds[keep_data + keep_aux]
        for c in ("year", "time", self.lat_name, self.lon_name):
            if c in out and c not in out.coords:
                out = out.set_coords(c)
        # normalize longitudes (do after set_coords)
        if self.lon_name in out.coords:
            out = self._normalize_dataset_longitude(out, self.lon_name)
        return out

    def _detect_time_coord(self, ds: xr.Dataset) -> str:
        if "year" in ds.coords or "year" in ds:
            return "year"
        if "time" in ds.coords or "time" in ds:
            return "time"
        raise ValueError("No 'year' or 'time' coordinate found.")

    @staticmethod
    def _compressible_encoding(ds, complevel=4):
        enc = {}
        for v in ds.data_vars:
            enc[v] = {"zlib": True, "complevel": complevel, "shuffle": True}
            if np.issubdtype(ds[v].dtype, np.floating) and ds[v].dtype == np.float32:
                enc[v]["dtype"] = "float32"
        return enc

    # ---------- Weights ----------
    def _find_area_field(self, ds: xr.Dataset):
        # Common names across E3SM/CMIP/MPAS
        for cand in ("area", "areacella", "areacello", "cell_area", "areaCell", "cellArea"):
            if cand in ds:
                return ds[cand]
        for cand in ("area", "areacella", "areacello"):
            if cand in ds.coords:
                return ds.coords[cand]
        return None

    def _get_weights(self, tmpl: Union[xr.Dataset, xr.DataArray], prefer="area"):
        lat = self.lat_name; lon = self.lon_name
        if isinstance(tmpl, xr.DataArray):
            ds = tmpl.to_dataset(name="_tmp_")
        else:
            ds = tmpl

        if lat not in ds.coords or lon not in ds.coords:
            return None

        if prefer == "area":
            area = self._find_area_field(ds)
            if area is not None:
                w = area
            else:
                prefer = "coslat"

        if prefer == "coslat":
            data_vars = [v for v in ds.data_vars]
            if not data_vars:
                return None
            w = np.cos(np.deg2rad(ds[lat]))
            base = ds[data_vars[0]]
            w, _ = xr.broadcast(w, base.isel({d: 0 for d in base.dims if d not in (lat, lon)}))
            w = w.squeeze(drop=True)

        w = xr.where(np.isfinite(w) & (w > 0), w, 0)
        return w

    def _weights_for_var(self, da: xr.DataArray, weights="auto"):
        if weights in ("auto", "area", "coslat"):
            prefer = "area" if weights in ("auto", "area") else "coslat"
            sample = da
            for d in list(sample.dims):
                if d not in (self.lat_name, self.lon_name):
                    sample = sample.isel({d: 0})
            return self._get_weights(sample, prefer=prefer)
        elif weights is None:
            return None
        else:
            raise ValueError("weights must be 'auto', 'area', 'coslat', or None")

    # ---------- Spatial helpers ----------
    def _coord_range(self, da):
        v = np.asarray(da.values)
        return float(np.nanmin(v)), float(np.nanmax(v))

    def _is_desc(self, da):
        vals = np.asarray(da.values)
        return vals[1] < vals[0] if vals.size > 1 else False

    def _interval_intersection(self, a0, a1, b0, b1):
        lo, hi = max(a0, b0), min(a1, b1)
        return (lo, hi) if lo <= hi else None

    def _subset_region(self, ds: xr.Dataset, regnam: str) -> xr.Dataset:
        """
        Dataset-aware regional subset:
        - uses normalized region defs (−180..180)
        - clips to dataset extents
        - supports descending latitude
        - handles wrap-around by concatenating segments
        """
        if regnam == "global":
            return ds

        (lat0, lat1), (lon0n, lon1n) = self.region_defs[regnam]
        lon0_raw, lon1_raw = self._region_meta[regnam]["lon_raw"]
        lon_segments = self._region_segments_lon(lon0n, lon1n, lon0_raw, lon1_raw)

        if self.lat_name not in ds.coords or self.lon_name not in ds.coords:
            raise ValueError(f"Dataset missing coords '{self.lat_name}'/'{self.lon_name}'")

        # Dataset extents (post-normalization in _preprocess)
        dlat0, dlat1 = self._coord_range(ds[self.lat_name])
        dlon0, dlon1 = self._coord_range(ds[self.lon_name])

        # Latitude intersection, respect coord order
        lat_int = self._interval_intersection(min(lat0, lat1), max(lat0, lat1), dlat0, dlat1)
        if lat_int is None:
            raise ValueError(f"No latitude overlap for region '{regnam}': "
                             f"requested [{lat0},{lat1}], dataset [{dlat0},{dlat1}]")
        la, lb = lat_int
        lat_desc = self._is_desc(ds[self.lat_name])
        lat_slice = slice(lb, la) if lat_desc else slice(la, lb)

        # Longitude intersection per segment
        segs = []
        for a, b in lon_segments:
            inter = self._interval_intersection(a, b, dlon0, dlon1)
            if inter is not None:
                segs.append(inter)

        if not segs:
            raise ValueError(f"No longitude overlap for region '{regnam}': "
                             f"requested {lon_segments}, dataset [{dlon0},{dlon1}]")

        # Warn if clipped
        if (la, lb) != (min(lat0, lat1), max(lat0, lat1)) or len(segs) != len(lon_segments):
            warnings.warn(f"Region '{regnam}' clipped to dataset extent: "
                          f"lat {la:.3f}..{lb:.3f}, lon segments {segs}")

        # Apply selections
        out = ds.sel({self.lat_name: lat_slice})
        parts = [out.sel({self.lon_name: slice(lo, hi)}) for lo, hi in segs]
        return parts[0] if len(parts) == 1 else xr.concat(parts, dim=self.lon_name)

    def _spatial_dims(self, da: xr.DataArray):
        return [d for d in da.dims if d in (self.lat_name, self.lon_name)]

    def _area_mean(self, da: xr.DataArray, weights="auto"):
        spdims = self._spatial_dims(da)
        if not spdims:
            return da  # non-spatial variable
        w = self._weights_for_var(da, weights=weights)
        if w is not None:
            w_sub = w
            if self.lat_name in w_sub.coords and self.lat_name in da.coords and not w_sub[self.lat_name].identical(da[self.lat_name]):
                w_sub = w_sub.sel({self.lat_name: da[self.lat_name]})
            if self.lon_name in w_sub.coords and self.lon_name in da.coords and not w_sub[self.lon_name].identical(da[self.lon_name]):
                w_sub = w_sub.sel({self.lon_name: da[self.lon_name]})
            w_eff = xr.where(np.isfinite(da), w_sub, 0)
            num = (da.where(np.isfinite(da), 0) * w_eff).sum(dim=spdims, skipna=False)
            den = w_eff.sum(dim=spdims)
            out = num / xr.where(den > 0, den, np.nan)
        else:
            out = da.mean(dim=spdims, skipna=True)
        return out
    
    def _is_valid_file(self, path, regions=("global",)):
        try:
            with xr.open_dataset(path, decode_times=False) as ds:
                if "region" in ds.dims and set(regions).issubset(set(ds.region.values)) and len(ds.data_vars) > 0:
                    return True
                else:
                    print(f"[WARN] File exists but missing expected content: {path}")
                    return False
        except Exception as e:
            print(f"[WARN] Unable to open or invalid NetCDF: {path} ({e})")
            return False
        
    def _apply_picontrol_drift_inplace(
        self,
        ds: xr.Dataset,
        ds_trend: xr.Dataset,
        tdim: str = "time",
        vars_to_correct=None,
        scale: str = "per_year",  # or "per_100yr" to match trend file
    ):
        """
        In-place: overwrite ds[v] with drift-corrected values using month-wise
        piControl trends from ds_trend.

        Assumes ds_trend has variables named '<v>_trend_month' with dims
        (month, lat, lon), where month=1..12.
        """
        if tdim not in ds.coords and tdim not in ds.dims:
            raise ValueError(f"Time dimension '{tdim}' not found in dataset.")

        t = ds[tdim]
        nt = t.size

        # Build time-rel (years since first step) and month index 1..12
        if hasattr(t, "dt"):
            # datetime-like (e.g., historical runs)
            time_year = t.dt.year + (t.dt.month - 0.5) / 12.0
            t_rel = time_year - float(time_year[0])
            month = t.dt.month  # 1..12
        else:
            # index-based monthly series (fallback)
            idx = xr.DataArray(
                np.arange(nt, dtype="float64"),
                dims=(tdim,),
                coords={tdim: t},
                name="time_index",
            )
            time_year = idx / 12.0
            t_rel = time_year - float(time_year[0])
            month = ((idx % 12) + 1).astype("int64")

        # Choose variables to correct
        if vars_to_correct is None:
            # all time-dependent vars that ALSO have a trend in ds_trend
            candidates = [v for v in ds.data_vars if tdim in ds[v].dims]
            vars_to_correct = []
            for v in candidates:
                trend_name = f"{v}_trend_month"
                if trend_name in ds_trend:
                    vars_to_correct.append(v)
        else:
            vars_to_correct = [
                v for v in vars_to_correct
                if (v in ds.data_vars) and (f"{v}_trend_month" in ds_trend)
            ]

        if not vars_to_correct:
            print("[WARN] No variables to drift-correct (vars_to_correct empty or trends missing).")
            return

        lat_name = self.lat_name
        lon_name = self.lon_name

        for v in vars_to_correct:
            trend_name = f"{v}_trend_month"
            if trend_name not in ds_trend:
                print(f"[WARN] No trend field {trend_name} in ds_trend; skipping {v}")
                continue

            slope = ds_trend[trend_name]  # dims: (month, lat, lon) ideally

            # Align slope's lat/lon with ds if needed
            for coord in (lat_name, lon_name):
                if coord in slope.coords and coord in ds.coords:
                    if not slope[coord].identical(ds[coord]):
                        slope = slope.sel({coord: ds[coord]})

            # Select slope for each time step by month index
            slope_t = slope.sel(month=month)  # broadcast to time, lat, lon

            # Build drift term
            if scale == "per_100yr":
                drift = slope_t * (t_rel / 100.0)
            else:  # per_year
                drift = slope_t * t_rel

            da_raw = ds[v]
            drift, da_raw = xr.align(drift, da_raw, join="inner")

            da_corr = da_raw - drift

            # Update attributes, keep same variable name
            attrs = dict(da_raw.attrs)
            ln = attrs.get("long_name", v)
            attrs["long_name"] = f"{ln} (piControl-drift corrected)"
            attrs["note"] = (
                "Linear, month-specific drift removed using 500-yr piControl trends "
                "from '<var>_trend_month'."
            )
            da_corr.attrs = attrs

            ds[v] = da_corr
            print(f"[INFO]  -> overwrote field {v} with drift-corrected values")

    def save_driftcorrected_fullfields_from_monthlies(
        self,
        trend_path,
        vars_to_correct=None,   # which base vars to de-drift; None -> auto
        time_slice=None,        # optional time subset
        force_compute=False,    # overwrite existing outputs?
        scale="per_year",       # must match trend file scale
        target_dir=None,        # where to put corrected files; default: self.out_dir
        compute=True,
        complevel=4,
        replace_original=False, # if True, overwrite v with v_driftcorr and drop suffix
    ):
        """
        For each monthly_base file:

          1) Apply piControl month-wise drift correction at each grid point,
             using trends in 'trend_path' (variables '<v>_trend_month').
             This adds '<v>_driftcorr' fields (and optionally replaces originals).

          2) Save the FULL corrected dataset to a new file in 'target_dir'
             with the SAME basename as the input file.

        Parameters
        ----------
        trend_path : str
            Path to the piControl trend file (with '<var>_trend_month').
        vars_to_correct : list or None
            If None, auto-detect all time-dependent vars that have
            corresponding '<v>_trend_month' in ds_trend.
        time_slice : slice or (start, end) or None
            Optional time selection applied before correction.
        force_compute : bool
            If False, skip writing if output file already exists.
        scale : {"per_year", "per_100yr"}
            Must match how trends were computed.
        target_dir : str or None
            Output directory. Defaults to self.out_dir.
        replace_original : bool
            If True, replace original variables with corrected ones
            (drop '_driftcorr' suffix). If False, keep both.
        """
        # --- Setup output dir
        out_dir = target_dir or self.out_dir
        Path(out_dir).mkdir(parents=True, exist_ok=True)

        # --- Open piControl trend dataset once
        ds_trend = xr.open_dataset(trend_path)

        for in_path in self._ensemble_paths():
            base_name = os.path.basename(in_path)
            out_path = os.path.join(out_dir, base_name)

            if (not force_compute) and os.path.exists(out_path):
                print(f"[SKIP] Drift-corrected full file already exists: {out_path}")
                continue

            print(f"[INFO] Creating drift-corrected full file: {out_path}")
            ds = xr.open_dataset(in_path)
            if self.chunks is not None:
                ds = ds.chunk(self.chunks)

            # Preprocess (coords + lon normalization, keep aux vars like area)
            ds = self._preprocess(ds)
            tdim = self._detect_time_coord(ds)

            # Optional time slicing
            if time_slice is not None:
                if isinstance(time_slice, slice):
                    ds = ds.sel({tdim: time_slice})
                elif isinstance(time_slice, (list, tuple)) and len(time_slice) == 2:
                    ds = ds.sel({tdim: slice(time_slice[0], time_slice[1])})
                else:
                    raise ValueError("time_slice must be a slice or (start, end)")

            # ---- Apply drift correction on the grid ----
            self._apply_picontrol_drift_inplace(
                ds,
                ds_trend,
                tdim=tdim,
                vars_to_correct=vars_to_correct,
                scale=scale,
            )
            # At this point ds contains v and v_driftcorr.

            if replace_original:
                # Replace v with v_driftcorr and drop the suffix
                to_drop = []
                to_add = {}
                for v in list(ds.data_vars):
                    if v.endswith("_driftcorr"):
                        base_v = v[:-10]  # strip "_driftcorr"
                        if base_v in ds:
                            to_drop.append(base_v)
                        new_da = ds[v].copy()
                        new_da.name = base_v
                        to_add[base_v] = new_da
                        to_drop.append(v)
                # Drop old vars and add replaced ones
                if to_drop:
                    ds = ds.drop_vars(list(set(to_drop)))
                for name, da in to_add.items():
                    ds[name] = da
                print("[INFO] Replaced original variables with drift-corrected versions.")

            # --- Compression (no special chunking; let backend handle)
            enc = self._compressible_encoding(ds, complevel=complevel)

            try:
                ds.to_netcdf(out_path, encoding=enc, compute=compute, engine="h5netcdf")
            except Exception:
                # Fallback: compute in-memory, then write
                ds = ds.unify_chunks().compute()
                ds.to_netcdf(out_path, encoding=enc, compute=True, engine="h5netcdf")

        return None

    def save_region_means_from_monthlies(
        self,
        regions=("global",),
        vars_to_average=None,      # None -> auto: all time-varying *_driftcorr
        time_slice=None,
        weights="auto",            # 'auto'|'area'|'coslat'|None
        include_nonspatial=True,   # True | "replicate" | False
        force_compute=False,
        compute=True,
        complevel=4,
        output_template="{base}.region_means.nc",
    ):
        """
        From ALREADY drift-corrected monthly files (full fields):

          - read each file from self.data_dir / file_template
          - pick variables to average (default: *_driftcorr that depend on time)
          - compute area-weighted regional/global means
          - save results into a NetCDF file per input file.

        This is similar to `save_driftcorrected_region_and_global_means_from_monthlies`,
        but assumes the drift correction has already been applied and stored in
        the input files.

        Parameters
        ----------
        regions : tuple of str
            Region names defined in self.region_defs, e.g. ("global", "Arctic").
        vars_to_average : list or None
            If None, auto-detect time-dependent variables ending with '_driftcorr'.
        time_slice : slice or (start, end) or None
            Optional time selection applied before averaging.
        weights : {"auto","area","coslat",None}
            How to build spatial weights.
        include_nonspatial : bool or "replicate"
            How to handle non-spatial vars.
        force_compute : bool
            If False, skip if output already exists and looks valid.
        output_template : str
            Pattern for output filenames, e.g. "{base}.region_means.nc".
        """
        Path(self.out_dir).mkdir(parents=True, exist_ok=True)

        bad_regions = [r for r in regions if r not in self.region_defs]
        if bad_regions:
            raise ValueError(
                f"Unknown regions: {bad_regions}. Available: {list(self.region_defs.keys())}"
            )

        for in_path in self._ensemble_paths():
            base = os.path.basename(in_path)
            if base.endswith(".nc"):
                base = base[:-3]

            ens_str = None
            m = re.search(r"\.en(\d{2})\.", base)
            if m:
                ens_str = m.group(1)

            out_fname = output_template
            if "{base}" in out_fname:
                out_fname = out_fname.replace("{base}", base)
            if "%(ENS)" in out_fname and ens_str:
                out_fname = out_fname.replace("%(ENS)", ens_str)

            out_path = os.path.join(self.out_dir, out_fname)

            if (not force_compute) and os.path.exists(out_path) and self._is_valid_file(out_path, regions):
                print(f"[SKIP] Region/global means already exist: {out_path}")
                continue

            print(f"[INFO] Creating region/global means from: {in_path}")
            ds = xr.open_dataset(in_path)
            if self.chunks is not None:
                ds = ds.chunk(self.chunks)

            ds = self._preprocess(ds)
            tdim = self._detect_time_coord(ds)

            # Optional time slicing
            if time_slice is not None:
                if isinstance(time_slice, slice):
                    ds = ds.sel({tdim: time_slice})
                elif isinstance(time_slice, (list, tuple)) and len(time_slice) == 2:
                    ds = ds.sel({tdim: slice(time_slice[0], time_slice[1])})
                else:
                    raise ValueError("time_slice must be a slice or (start, end)")

            # --------- pick variables to average ----------
            if vars_to_average is None:
                # Default: all time-dependent variables
                vars_to_average = [
                    v for v in ds.data_vars
                    if (tdim in ds[v].dims)
                ]
            else:
                vars_to_average = [
                    v for v in vars_to_average
                    if (v in ds.data_vars) and (tdim in ds[v].dims)
                ]

            if not vars_to_average:
                print(f"[WARN] No variables selected for region means in {in_path}; skipping.")
                continue

            keep_aux = [v for v in self.keep_vars if v in ds]
            ds = ds[vars_to_average + keep_aux]

            reg_series = {}
            for reg in regions:
                # 1) subset region once
                ds_reg = ds if reg == "global" else self._subset_region(ds, reg)
                ds_reg = ds_reg.persist()

                out_vars = {}

                # 2) build weights ONCE per region using first spatial var
                w_reg = None
                for _v in vars_to_average:
                    spdims = self._spatial_dims(ds_reg[_v])
                    if spdims:
                        w_reg = self._weights_for_var(ds_reg[_v], weights=weights)
                        break

                # 3) loop vars, reuse ds_reg & w_reg
                for v in vars_to_average:
                    da = ds_reg[v]
                    spdims = self._spatial_dims(da)

                    if spdims:
                        if w_reg is not None:
                            w_use = w_reg
                            for d in (self.lat_name, self.lon_name):
                                if (d in da.coords) and (d in w_use.coords) and not w_use[d].identical(da[d]):
                                    w_use = w_use.sel({d: da[d]})
                            area_mean = da.weighted(
                                xr.where(np.isfinite(da), w_use, 0)
                            ).mean(dim=spdims)
                        else:
                            area_mean = da.mean(dim=spdims, skipna=True)

                        attrs = dict(da.attrs)
                        ln = attrs.get("long_name", v)
                        attrs["long_name"] = f"Area-averaged {ln} ({reg})"
                        out_vars[v] = area_mean.assign_attrs(attrs)
                    else:
                        if not include_nonspatial:
                            continue
                        if include_nonspatial == "replicate":
                            if reg == regions[0]:
                                out_vars[v] = da
                        else:
                            out_vars[v] = da

                dso = xr.Dataset(out_vars).expand_dims(region=[reg])
                reg_series[reg] = dso

            # 4) concat across regions
            out_ds = xr.concat([reg_series[r] for r in regions], dim="region")

            # 5) replicate non-spatial across region, if requested
            if include_nonspatial == "replicate":
                for v in list(out_ds.data_vars):
                    if "region" not in out_ds[v].dims:
                        out_ds[v] = out_ds[v].expand_dims(region=out_ds.region)

            # 6) compression & chunking
            enc = self._compressible_encoding(out_ds, complevel=complevel)
            for v, e in enc.items():
                dims = out_ds[v].dims
                cs = []
                for d in dims:
                    if d == "time":
                        cs.append(out_ds.sizes[d])
                    elif d == "region":
                        cs.append(out_ds.sizes[d])
                    elif d in (self.lat_name, self.lon_name):
                        target = 180
                        cs.append(min(out_ds.sizes[d], target))
                    else:
                        cs.append(min(out_ds.sizes[d], 512))
                e["chunksizes"] = tuple(cs)

            try:
                out_ds.to_netcdf(out_path, compute=compute, encoding=enc, engine="h5netcdf")
            except Exception:
                out_ds = out_ds.unify_chunks().compute()
                out_ds.to_netcdf(out_path, compute=True, encoding=enc, engine="h5netcdf")

        return None

            

In [ ]:
if __name__ == "__main__":
    # --- Paths
    raw_dir        = f"{GLOBAL_DIR}"                # original monthly_base files
    drift_dir      = f"{GLOBAL_DRIFTCORR_DIR}"      # full, drift-corrected fields
    regmn_dir      = f"{REGMN_DRIFTCORR_DIR}"       # regional means of drift-corrected fields
    Path(drift_dir).mkdir(parents=True, exist_ok=True)
    Path(regmn_dir).mkdir(parents=True, exist_ok=True)

    # --- piControl trend file (already computed with MonthlyBaseProcessor.save_linear_trend)
    trend_path = (
        f"{GLOBAL_DIR}/"
        "v3.LR.piControl.en00.trend_500yr_monthly.000101-050012.nc"
    )

    # --- Vars & ensemble
    var_dict  = read_var_dict()
    group     = "v3.LR.historical"
    nens      = 25                         # en00..en24 or en01..en25, auto-detected
    period    = "1850-2050"
    start_year, end_year = period.split("-")
    date_span = f"{start_year}01-{end_year}12"

    file_template = f"{group}.en%(ENS).monthly_base.{date_span}.nc"

    # Regions, weights, etc.
    regions      = ("global", "Arctic", "Arctic2")
    time_slice   = None
    weights      = "coslat"
    chunks       = {"time": -1}   # single time chunk; OK if memory allows
    keep_vars    = ["area"]
    lat_name     = "lat"
    lon_name     = "lon"
    
    include_nonspatial = True
    compute            = True
    complevel          = 4
    force_compute      = True
    replace_original   = True 
    
    vars_to_correct = None   # or e.g. ["SST", "TS", "FSNT", "FLNT", "TURFLX"]
    scale           = "per_year"   # must match the scale used in save_linear_trend
    vars_to_average = None           # auto: all time-varying *_driftcorr

    # ------------------------------------------------------------------
    # STEP 1: start from RAW monthly_base files -> drift-corrected FULL FIELDS
    # ------------------------------------------------------------------
    proc_raw = EnsembleDataProcs(
        data_dir=raw_dir,                # where monthly_base files actually are
        file_template=file_template,
        num_ens=nens,
        var_dict=var_dict,
        out_dir=drift_dir,               # where drift-corrected full files go
        chunks=chunks,
        keep_vars=keep_vars,
        lat_name=lat_name,
        lon_name=lon_name,
    )

    print("[STEP 1] Creating drift-corrected full fields...")
    proc_raw.save_driftcorrected_fullfields_from_monthlies(
        trend_path=trend_path,
        vars_to_correct=vars_to_correct,
        time_slice=time_slice,
        force_compute=force_compute,            # overwrite if needed
        scale=scale,
        target_dir=drift_dir,          # explicit, even though out_dir=drift_dir
        compute=compute,
        complevel=complevel,
        replace_original=replace_original,        # keep both v and v_driftcorr in the file
    )


[STEP 1] Creating drift-corrected full fields...
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/global v3.LR.historical.en00.monthly_base.185001-205012.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/global v3.LR.historical.en01.monthly_base.185001-205012.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/global v3.LR.historical.en02.monthly_base.185001-205012.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/global v3.LR.historical.en03.monthly_base.185001-205012.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/global v3.LR.historical.en04.monthly_base.185001-205012.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/global v3.LR.historical.en05.monthly_base.185001-205012.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/global v3.LR.historical.en06.monthly_base.185001-205012.nc
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_pro

[INFO]  -> overwrote field SICONC with drift-corrected values
[INFO]  -> overwrote field TS with drift-corrected values
[INFO]  -> overwrote field TURFLX with drift-corrected values
[INFO]  -> overwrote field LHFLX with drift-corrected values
[INFO]  -> overwrote field SHFLX with drift-corrected values
[INFO]  -> overwrote field FLUT with drift-corrected values
[INFO]  -> overwrote field FLNT with drift-corrected values
[INFO]  -> overwrote field FSNT with drift-corrected values
[INFO]  -> overwrote field FSNTOA with drift-corrected values
[INFO]  -> overwrote field FLNTC with drift-corrected values
[INFO]  -> overwrote field FSNTC with drift-corrected values
[INFO]  -> overwrote field FLUTC with drift-corrected values
[INFO]  -> overwrote field FSNTOAC with drift-corrected values
[INFO]  -> overwrote field RADNET with drift-corrected values
[INFO]  -> overwrote field FSNS with drift-corrected values
[INFO]  -> overwrote field FSDS with drift-corrected values
[INFO]  -> overwrote field

[INFO]  -> overwrote field FLNS_imsk with drift-corrected values
[INFO]  -> overwrote field FLUS_imsk with drift-corrected values
[INFO]  -> overwrote field FLNSC_imsk with drift-corrected values
[INFO]  -> overwrote field FLUSC_imsk with drift-corrected values
[INFO]  -> overwrote field FLDSC_imsk with drift-corrected values
[INFO]  -> overwrote field SWCF_SRF_imsk with drift-corrected values
[INFO]  -> overwrote field LWCF_SRF_imsk with drift-corrected values
[INFO]  -> overwrote field CRE_SRF_imsk with drift-corrected values
[INFO]  -> overwrote field SWCF_imsk with drift-corrected values
[INFO]  -> overwrote field LWCF_imsk with drift-corrected values
[INFO]  -> overwrote field CRE_imsk with drift-corrected values
[INFO]  -> overwrote field ALBEDO_SRF_imsk with drift-corrected values
[INFO]  -> overwrote field ALBEDOC_SRF_imsk with drift-corrected values
[INFO] Replaced original variables with drift-corrected versions.
[INFO] Creating drift-corrected full file: /lcrc/group/e3sm/ac.

[INFO]  -> overwrote field LHFLX_imsk with drift-corrected values
[INFO]  -> overwrote field SHFLX_imsk with drift-corrected values
[INFO]  -> overwrote field FLUT_imsk with drift-corrected values
[INFO]  -> overwrote field FLNT_imsk with drift-corrected values
[INFO]  -> overwrote field FSNT_imsk with drift-corrected values
[INFO]  -> overwrote field FSNTOA_imsk with drift-corrected values
[INFO]  -> overwrote field FLNTC_imsk with drift-corrected values
[INFO]  -> overwrote field FSNTC_imsk with drift-corrected values
[INFO]  -> overwrote field FLUTC_imsk with drift-corrected values
[INFO]  -> overwrote field FSNTOAC_imsk with drift-corrected values
[INFO]  -> overwrote field RADNET_imsk with drift-corrected values
[INFO]  -> overwrote field FSNS_imsk with drift-corrected values
[INFO]  -> overwrote field FSDS_imsk with drift-corrected values
[INFO]  -> overwrote field FSUS_imsk with drift-corrected values
[INFO]  -> overwrote field FSNSC_imsk with drift-corrected values
[INFO]  -> o

[INFO]  -> overwrote field FSUSC with drift-corrected values
[INFO]  -> overwrote field FLDS with drift-corrected values
[INFO]  -> overwrote field FLNS with drift-corrected values
[INFO]  -> overwrote field FLUS with drift-corrected values
[INFO]  -> overwrote field FLNSC with drift-corrected values
[INFO]  -> overwrote field FLUSC with drift-corrected values
[INFO]  -> overwrote field FLDSC with drift-corrected values
[INFO]  -> overwrote field SWCF_SRF with drift-corrected values
[INFO]  -> overwrote field LWCF_SRF with drift-corrected values
[INFO]  -> overwrote field CRE_SRF with drift-corrected values
[INFO]  -> overwrote field SWCF with drift-corrected values
[INFO]  -> overwrote field LWCF with drift-corrected values
[INFO]  -> overwrote field CRE with drift-corrected values
[INFO]  -> overwrote field ALBEDO_SRF with drift-corrected values
[INFO]  -> overwrote field ALBEDOC_SRF with drift-corrected values
[INFO]  -> overwrote field SST_imsk with drift-corrected values
[INFO]  -

[INFO]  -> overwrote field SST with drift-corrected values
[INFO]  -> overwrote field SICONC with drift-corrected values
[INFO]  -> overwrote field TS with drift-corrected values
[INFO]  -> overwrote field TURFLX with drift-corrected values
[INFO]  -> overwrote field LHFLX with drift-corrected values
[INFO]  -> overwrote field SHFLX with drift-corrected values
[INFO]  -> overwrote field FLUT with drift-corrected values
[INFO]  -> overwrote field FLNT with drift-corrected values
[INFO]  -> overwrote field FSNT with drift-corrected values
[INFO]  -> overwrote field FSNTOA with drift-corrected values
[INFO]  -> overwrote field FLNTC with drift-corrected values
[INFO]  -> overwrote field FSNTC with drift-corrected values
[INFO]  -> overwrote field FLUTC with drift-corrected values
[INFO]  -> overwrote field FSNTOAC with drift-corrected values
[INFO]  -> overwrote field RADNET with drift-corrected values
[INFO]  -> overwrote field FSNS with drift-corrected values
[INFO]  -> overwrote field 

[INFO]  -> overwrote field FLDS_imsk with drift-corrected values
[INFO]  -> overwrote field FLNS_imsk with drift-corrected values
[INFO]  -> overwrote field FLUS_imsk with drift-corrected values
[INFO]  -> overwrote field FLNSC_imsk with drift-corrected values
[INFO]  -> overwrote field FLUSC_imsk with drift-corrected values
[INFO]  -> overwrote field FLDSC_imsk with drift-corrected values
[INFO]  -> overwrote field SWCF_SRF_imsk with drift-corrected values
[INFO]  -> overwrote field LWCF_SRF_imsk with drift-corrected values
[INFO]  -> overwrote field CRE_SRF_imsk with drift-corrected values
[INFO]  -> overwrote field SWCF_imsk with drift-corrected values
[INFO]  -> overwrote field LWCF_imsk with drift-corrected values
[INFO]  -> overwrote field CRE_imsk with drift-corrected values
[INFO]  -> overwrote field ALBEDO_SRF_imsk with drift-corrected values
[INFO]  -> overwrote field ALBEDOC_SRF_imsk with drift-corrected values
[INFO] Replaced original variables with drift-corrected versions

[INFO]  -> overwrote field TURFLX_imsk with drift-corrected values
[INFO]  -> overwrote field LHFLX_imsk with drift-corrected values
[INFO]  -> overwrote field SHFLX_imsk with drift-corrected values
[INFO]  -> overwrote field FLUT_imsk with drift-corrected values
[INFO]  -> overwrote field FLNT_imsk with drift-corrected values
[INFO]  -> overwrote field FSNT_imsk with drift-corrected values
[INFO]  -> overwrote field FSNTOA_imsk with drift-corrected values
[INFO]  -> overwrote field FLNTC_imsk with drift-corrected values
[INFO]  -> overwrote field FSNTC_imsk with drift-corrected values
[INFO]  -> overwrote field FLUTC_imsk with drift-corrected values
[INFO]  -> overwrote field FSNTOAC_imsk with drift-corrected values
[INFO]  -> overwrote field RADNET_imsk with drift-corrected values
[INFO]  -> overwrote field FSNS_imsk with drift-corrected values
[INFO]  -> overwrote field FSDS_imsk with drift-corrected values
[INFO]  -> overwrote field FSUS_imsk with drift-corrected values
[INFO]  -> 

[INFO]  -> overwrote field FSDSC with drift-corrected values
[INFO]  -> overwrote field FSUSC with drift-corrected values
[INFO]  -> overwrote field FLDS with drift-corrected values
[INFO]  -> overwrote field FLNS with drift-corrected values
[INFO]  -> overwrote field FLUS with drift-corrected values
[INFO]  -> overwrote field FLNSC with drift-corrected values
[INFO]  -> overwrote field FLUSC with drift-corrected values
[INFO]  -> overwrote field FLDSC with drift-corrected values
[INFO]  -> overwrote field SWCF_SRF with drift-corrected values
[INFO]  -> overwrote field LWCF_SRF with drift-corrected values
[INFO]  -> overwrote field CRE_SRF with drift-corrected values
[INFO]  -> overwrote field SWCF with drift-corrected values
[INFO]  -> overwrote field LWCF with drift-corrected values
[INFO]  -> overwrote field CRE with drift-corrected values
[INFO]  -> overwrote field ALBEDO_SRF with drift-corrected values
[INFO]  -> overwrote field ALBEDOC_SRF with drift-corrected values
[INFO]  -> o